# Train + evaluate DistilBERT (free Colab GPU)
**Runtime → Change runtime type → T4 GPU** (free). Do not pick a paid GPU, and ignore any Colab Pro / compute-unit offers.

Before running: upload `train.csv`, `val.csv`, `test.csv` from your Mac's `data/processed/` to
`MyDrive/airbnb_nlp/data/processed/` in Google Drive.

Checkpoints go to Drive every epoch. If Colab disconnects, reconnect and run all cells again: training resumes.

In [ ]:
# 1. Check the GPU and Python version
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys; print(sys.version)

In [ ]:
# 2. Mount Google Drive (free) and set paths
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/airbnb_nlp'
DATA = f'{DRIVE}/data/processed'
MODELS = f'{DRIVE}/models/distilbert'
REPORTS = f'{DRIVE}/reports'
import os
print(sorted(os.listdir(DATA)))   # should show test.csv, train.csv, val.csv

In [ ]:
# 3. Get the code from GitHub (same commit as your Mac) and install it
!rm -rf /content/repo
!git clone -q -b nlp-finetuning https://github.com/sgaind-data/Airbnb_Review_Scores_ML.git /content/repo
%cd /content/repo
!git log --oneline -1
!pip install -q -e ".[train]"

In [ ]:
# 4. Smoke test: 1 epoch on 500 rows (about 1 minute). Catches errors before the real run.
!python -m airbnb_nlp.train --data-dir {DATA} --output-dir /content/smoke --epochs 1 --limit 500

In [ ]:
# 5. Full training: 3 epochs, checkpoints saved to Drive (about 10-20 minutes on a T4)
!python -m airbnb_nlp.train --data-dir {DATA} --output-dir {MODELS}

In [ ]:
# 6. Test-set evaluation vs VADER, confusion matrix, error analysis
!python -m airbnb_nlp.evaluate --data-dir {DATA} --model-dir {MODELS}/final --reports-dir {REPORTS}
!cp {MODELS}/train_history.json {REPORTS}/

In [ ]:
# 7. Look at the results
from IPython.display import Image, Markdown, display
import pandas as pd
display(Markdown(open(f'{REPORTS}/comparison.md').read()))
display(Image(f'{REPORTS}/confusion_matrix.png'))
pd.set_option('display.max_colwidth', 300)
pd.read_csv(f'{REPORTS}/error_analysis.csv')